In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

RANDOM_SEED   = 42
LEARNING_RATE = 0.001
BATCH_SIZE    = 256
EPOCHS_CNN    = 30
EPOCHS_AE     = 30
device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

CLASS_MAP = {
    'normal'                        : 'Normal',
    'anomalous(DoSattack)'          : 'DoS',
    'anomalous(scan)'               : 'Scan',
    'anomalous(malitiousControl)'   : 'MaliciousControl',
    'anomalous(malitiousOperation)' : 'MaliciousOperation',
    'anomalous(spying)'             : 'Spying',
    'anomalous(dataProbing)'        : 'DataProbing',
    'anomalous(wrongSetUp)'         : 'WrongSetUp',
}
CLASS_TO_INT = {v: i for i, v in enumerate(CLASS_MAP.values())}
INT_TO_CLASS = {v: k for k, v in CLASS_TO_INT.items()}

SEARCH_PATHS = [
    Path("DS2OS.csv"),
    Path.home() / "Downloads" / "DS2OS.csv",
    (Path.home() / ".cache" / "kagglehub" / "datasets" / "libamariyam" / "ds2os-dataset" / "versions" / "1" / "DS2OS.csv"),
]

CSV_PATH = next((p for p in SEARCH_PATHS if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError("DS2OS.csv no encontrado.")

df_raw = pd.read_csv(CSV_PATH)
print(f"Shape: {df_raw.shape}")

Shape: (357952, 13)


In [2]:
df = df_raw.copy()

df['accessedNodeType'] = df['accessedNodeType'].fillna('Malicious')
df['value'] = df['value'].replace({'False': 0, 'True': 1, 'Twenty': 20, 'none': 0})
df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0)
df = df.drop(columns=['timestamp'])
df['y'] = df['normality'].map(CLASS_MAP).map(CLASS_TO_INT).astype(np.int8)
df = df.drop(columns=['normality'])

CAT_COLS = [
    'sourceID', 'sourceAddress', 'sourceType', 'sourceLocation',
    'destinationServiceAddress', 'destinationServiceType',
    'destinationLocation', 'accessedNodeAddress', 'accessedNodeType', 'operation',
]
for col in CAT_COLS:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

FEATURE_COLS = CAT_COLS + ['value']
X = df[FEATURE_COLS].values.astype(np.float32)
y = df['y'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

y_train_bin = (y_train > 0).astype(int)
y_test_bin  = (y_test  > 0).astype(int)

X_train_cnn    = X_train
y_train_multi  = y_train
X_train_autoenc = X_train[y_train == 0]

N_CLASSES = len(np.unique(y_train_multi))
INPUT_DIM = X_train_cnn.shape[1]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"AE Train (Normal): {len(X_train_autoenc):,}")
print(f"INPUT_DIM: {INPUT_DIM} | N_CLASSES: {N_CLASSES}")

Train: 286,361 | Test: 71,591
AE Train (Normal): 278,347
INPUT_DIM: 11 | N_CLASSES: 8


In [3]:
class DPACK(nn.Module):
    def __init__(self, input_dim, n_classes):
        super(DPACK, self).__init__()
        self.input_dim = input_dim
        self.conv1 = nn.Sequential(nn.Conv1d(1, 32, kernel_size=6, stride=1, padding=5), nn.ReLU(), nn.BatchNorm1d(32))
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Sequential(nn.Conv1d(32, 64, kernel_size=6, stride=1, padding=5), nn.ReLU(), nn.BatchNorm1d(64))
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)
        self._conv_out_size = self._get_conv_output_size()
        self.fc5 = nn.Sequential(nn.Linear(self._conv_out_size, 1024), nn.ReLU(), nn.BatchNorm1d(1024))
        self.fc6 = nn.Sequential(nn.Linear(1024, 25), nn.ReLU(), nn.BatchNorm1d(25))
        self.fc7 = nn.Linear(25, n_classes)
        self.ae_fc8  = nn.Sequential(nn.Linear(1024, 512), nn.ReLU())
        self.ae_fc9  = nn.Sequential(nn.Linear(512,  256), nn.ReLU())
        self.ae_fc10 = nn.Sequential(nn.Linear(256,  512), nn.ReLU())
        self.ae_fc11 = nn.Sequential(nn.Linear(512, 1024), nn.ReLU())
        self.ae_out  = nn.Linear(1024, input_dim)

    def _get_conv_output_size(self):
        with torch.no_grad():
            dummy = torch.zeros(1, 1, self.input_dim)
            out   = self.pool2(self.conv2(self.pool1(self.conv1(dummy))))
            return int(out.numel())

    def forward(self, x):
        h = x.unsqueeze(1)
        h = self.pool1(self.conv1(h))
        h = self.pool2(self.conv2(h))
        h = h.view(h.size(0), -1)
        h5      = self.fc5(h)
        cnn_out = self.fc7(self.fc6(h5))
        ae_out  = self.ae_out(self.ae_fc11(self.ae_fc10(self.ae_fc9(self.ae_fc8(h5)))))
        return cnn_out, ae_out, h5


def train_cnn_phase(model, X_tr, y_tr, epochs, batch_size, lr, device):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    loader    = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)),
                           batch_size=batch_size, shuffle=True, num_workers=0)
    for epoch in range(epochs):
        total_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            cnn_out, ae_out, _ = model(X_batch)
            loss = criterion(cnn_out, y_batch) + nn.MSELoss()(ae_out, X_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct    += (cnn_out.argmax(dim=1) == y_batch).sum().item()
            total      += y_batch.size(0)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoca {epoch+1:>3}/{epochs} | Loss: {total_loss/len(loader):.4f} | Acc: {100*correct/total:.2f}%")
    return model


def train_autoencoder_phase(model, X_ae, epochs, batch_size, lr, device):
    model.train()
    ae_params = (list(model.fc5.parameters())    + list(model.ae_fc8.parameters()) +
                 list(model.ae_fc9.parameters()) + list(model.ae_fc10.parameters()) +
                 list(model.ae_fc11.parameters()) + list(model.ae_out.parameters()))
    optimizer = optim.Adam(ae_params, lr=lr)
    criterion = nn.MSELoss()
    loader    = DataLoader(TensorDataset(torch.FloatTensor(X_ae)),
                           batch_size=batch_size, shuffle=True, num_workers=0)
    for epoch in range(epochs):
        total_mse = 0.0
        for (X_batch,) in loader:
            X_batch = X_batch.to(device)
            optimizer.zero_grad()
            _, ae_out, _ = model(X_batch)
            loss = criterion(ae_out, X_batch)
            loss.backward()
            optimizer.step()
            total_mse += loss.item()
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoca {epoch+1:>3}/{epochs} | MSE: {total_mse/len(loader):.6f}")
    return model


def compute_threshold(model, X_benign, device, batch_size=512):
    model.eval()
    mse_list = []
    with torch.no_grad():
        for i in range(0, len(X_benign), batch_size):
            X_batch = torch.FloatTensor(X_benign[i:i+batch_size]).to(device)
            _, ae_out, _ = model(X_batch)
            mse_list.extend(((ae_out - X_batch) ** 2).mean(dim=1).cpu().numpy())
    mse_arr = np.array(mse_list)
    max_mse, p99, std = np.max(mse_arr), np.percentile(mse_arr, 99), np.std(mse_arr)
    threshold = p99 if (max_mse - p99) > 3 * std else max_mse
    print(f"  Umbral = {threshold:.6f} ({'percentil 99' if threshold == p99 else 'max MSE'})")
    return threshold


torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

model = DPACK(input_dim=INPUT_DIM, n_classes=N_CLASSES).to(device)
print(f"Parametros totales: {sum(p.numel() for p in model.parameters()):,}")

print("\nFASE 1 - CNN")
model = train_cnn_phase(model, X_train_cnn, y_train_multi, EPOCHS_CNN, BATCH_SIZE, LEARNING_RATE, device)

print("\nFASE 2 - Autoencoder")
model = train_autoencoder_phase(model, X_train_autoenc, EPOCHS_AE, BATCH_SIZE, LEARNING_RATE, device)

print("\nUMBRAL")
threshold = compute_threshold(model, X_train_autoenc, device)

Parametros totales: 1,759,238

FASE 1 - CNN
  Epoca   1/30 | Loss: 0.2637 | Acc: 96.58%
  Epoca   5/30 | Loss: 0.0169 | Acc: 99.32%
  Epoca  10/30 | Loss: 0.0143 | Acc: 99.36%
  Epoca  15/30 | Loss: 0.0144 | Acc: 99.36%
  Epoca  20/30 | Loss: 0.0140 | Acc: 99.37%
  Epoca  25/30 | Loss: 0.0133 | Acc: 99.39%
  Epoca  30/30 | Loss: 0.0137 | Acc: 99.39%

FASE 2 - Autoencoder
  Epoca   1/30 | MSE: 0.000961
  Epoca   5/30 | MSE: 0.000841
  Epoca  10/30 | MSE: 0.000912
  Epoca  15/30 | MSE: 0.000885
  Epoca  20/30 | MSE: 0.000805
  Epoca  25/30 | MSE: 0.000479
  Epoca  30/30 | MSE: 0.000385

UMBRAL
  Umbral = 0.000863 (percentil 99)


In [4]:
def predict(model, X, threshold, device, batch_size=512):
    model.eval()
    mse_list = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            X_batch = torch.FloatTensor(X[i:i+batch_size]).to(device)
            _, ae_out, _ = model(X_batch)
            mse_list.extend(((ae_out - X_batch) ** 2).mean(dim=1).cpu().numpy())
    return (np.array(mse_list) > threshold).astype(int)


def compute_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    accuracy  = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    far       = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr       = fn / (tp + fn) if (tp + fn) > 0 else 0.0
    return {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall,
            'F1': f1, 'FAR': far, 'FNR': fnr, 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn}


y_pred  = predict(model, X_test, threshold, device)
metrics = compute_metrics(y_test_bin, y_pred)

SEP = '=' * 50
print("D-PACK - Dataset DS2OS")
print(SEP)
for k in ('Accuracy', 'Precision', 'Recall', 'F1', 'FAR', 'FNR'):
    print(f"  {k:<12}: {metrics[k]*100:.2f}%")
print(SEP)
print(f"  TP={int(metrics['TP']):,}  TN={int(metrics['TN']):,}  FP={int(metrics['FP']):,}  FN={int(metrics['FN']):,}")

print("\nRecall por clase:")
for cls_id, cls_name in INT_TO_CLASS.items():
    mask         = (y_test == cls_id)
    expected_bin = 0 if cls_id == 0 else 1
    rec = ((y_pred == expected_bin) & mask).sum() / mask.sum() if mask.sum() > 0 else 0.0
    print(f"  {cls_name:<22}: {rec*100:.1f}%  ({mask.sum():,} instancias)")

D-PACK - Dataset DS2OS
  Accuracy    : 98.49%
  Precision   : 80.82%
  Recall      : 60.36%
  F1          : 69.11%
  FAR         : 0.41%
  FNR         : 39.64%
  TP=1,209  TN=69,301  FP=287  FN=794

Recall por clase:
  Normal                : 99.6%  (69,588 instancias)
  DoS                   : 31.3%  (1,156 instancias)
  Scan                  : 100.0%  (310 instancias)
  MaliciousControl      : 100.0%  (178 instancias)
  MaliciousOperation    : 100.0%  (161 instancias)
  Spying                : 100.0%  (106 instancias)
  DataProbing           : 100.0%  (68 instancias)
  WrongSetUp            : 100.0%  (24 instancias)
